In [2]:
import json, os
from openai import OpenAI
from pydantic import BaseModel
import dotenv
dotenv.load_dotenv(os.path.expanduser("~/missing_object/.env"))

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

with open("classes.json") as f:
    classes = json.load(f)

In [3]:
# Merge similar lowest-level classes
merges = {
    "anti_aesthetics": {
        "clarity_and_focus": {
            "compression_artifacts": ["compression_artifacts", "compression_mosquito_noise"],
            "blur_and_defocus": ["blurriness", "out_of_focus"],
            "digital_corruption": ["datamosh", "stuttered_frame_corruption"],
            "analog_decay": ["vhs_decay", "analog_decay_texture"],
            "low_res_display": ["low_resolution_retro", "scanline_texture"],
        },
        "color_quality": {
            "unhealthy_color_cast": ["sodium_vapor_cast", "sickly_color_cast"],
        },
        "emotion": {
            "uncanny_dream_unease": ["nostalgic_unease", "disorienting_dream_calm"],
        },
        "context_and_setting": {
            "infinite_recursive_space": ["backrooms_infinite_interior", "endless_corridor_depth"],
        },
        # Cross-category merge: liminal_space_aesthetic (realism_and_style) + liminal_public_space (context_and_setting)
        "realism_and_style": {
            "liminal_space": ["liminal_space_aesthetic"],
        },
    }
}

# Also remove liminal_public_space from context_and_setting (merged into liminal_space above)
cross_category_removes = {
    "anti_aesthetics": {
        "context_and_setting": ["liminal_public_space"],
        "structure_and_perspective": ["endless_corridor_depth"],  # merged into context_and_setting
    }
}

def apply_merges(classes, merges, cross_removes):
    merged = json.loads(json.dumps(classes))  # deep copy
    
    for top_key, categories in merges.items():
        for cat_key, merge_groups in categories.items():
            cat = merged[top_key][cat_key]
            for new_name, old_names in merge_groups.items():
                # Combine descriptions
                descs = [cat[k] for k in old_names if k in cat]
                # Also check other categories for cross-category merges
                if not descs:
                    for other_cat in merged[top_key].values():
                        descs.extend(other_cat[k] for k in old_names if k in other_cat)
                combined = " ".join(descs)
                # Remove old keys
                for k in old_names:
                    cat.pop(k, None)
                cat[new_name] = combined
    
    for top_key, categories in cross_removes.items():
        for cat_key, keys in categories.items():
            for k in keys:
                merged[top_key][cat_key].pop(k, None)
    
    return merged

merged_classes = apply_merges(classes, merges, cross_category_removes)

# Print summary
for top in merged_classes:
    total = sum(len(v) for v in merged_classes[top].values())
    print(f"{top}: {total} classes")
    for cat, items in merged_classes[top].items():
        print(f"  {cat}: {list(items.keys())}")

anti_aesthetics: 64 classes
  realism_and_style: ['uncanny_valley', 'plastic_toy_look', 'weirdcore_fragmented_identity', 'analog_horror_found_footage', 'Psychedelic art', 'outsider_naive_style', 'liminal_space']
  color_quality: ['wrong_object_color', 'uneven_white_balance', 'clashing_disharmony', 'muted_color', 'color_bleeding', 'chromatic_aberration', 'color_banding', 'retro_faded_palette', 'toxic_neon_palette', 'monotone_tint_overcast', 'unhealthy_color_cast']
  clarity_and_focus: ['motion_blur', 'noise', 'double_exposure', 'heavy_grain', 'over_sharpened_haloing', 'compression_artifacts', 'blur_and_defocus', 'digital_corruption', 'analog_decay', 'low_res_display']
  emotion: ['negative_personal_emotion', 'atmospheric_distress', 'depersonalization_detachment', 'uncanny_dream_unease']
  distortion: ['melted_objects', 'distorted_sense', 'anatomical_deformity', 'non_euclidean_geometry', 'recursive_space_repetition', 'facial_feature_displacement']
  execution_quality: ['unfinished', 'old

In [4]:
# Save merged classes
with open("classes_merged.json", "w") as f:
    json.dump(merged_classes, f, indent=4)

In [5]:
# Define return type — a list of 10 prompt set objects
from typing import List

class PromptSet(BaseModel):
    image_prompt: str    # Full image generation prompt
    negative_prompt: str # What to avoid (no negation words)
    object_prompt: str   # Same scene as image_prompt, stripped of all effects/style
    effects_prompt: str  # Same effects as image_prompt, stripped of all objects/content

class ClassPrompts(BaseModel):
    prompts: List[PromptSet]  # Exactly 10 prompt sets

# Flatten all classes into a list of (top_category, sub_category, class_name, description)
flat_classes = []
for top_key, categories in merged_classes.items():
    for cat_key, items in categories.items():
        for class_name, description in items.items():
            flat_classes.append((top_key, cat_key, class_name, description))

print(f"Total classes to process: {len(flat_classes)}")

Total classes to process: 99


In [6]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import tqdm

MODEL = "gpt-5.4"
RESULTS_PATH = "generated_prompts.jsonl"

# Load already-completed classes
done = set()
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            done.add(f"{obj['top_category']}/{obj['sub_category']}/{obj['class_name']}")
print(f"Already done: {len(done)}/{len(flat_classes)}")

def generate_prompts(top_key, cat_key, class_name, description):
    system = """You are an expert at writing image generation prompts (for Stable Diffusion / FLUX / DALL-E).
Given an aesthetic class, generate exactly 10 diverse PromptSet objects. Each PromptSet has four fields that all describe the same specific scene:
- image_prompt: the full generation prompt for a scene clearly exhibiting this aesthetic.
- negative_prompt: what to avoid so the aesthetic is clearly present. Do NOT use negation words ("no X", "avoid X") — just list the unwanted qualities directly (e.g. "sharp focus, high resolution").
- object_prompt: the same scene as image_prompt but with ALL style, effects, and mood words removed — only physical subjects and setting.
- effects_prompt: the same effects present in image_prompt but with ALL objects and content removed — only visual effects, style, and rendering qualities.

IMPORTANT: 
- Every prompt must be 2 sentences or shorter, besides the effect prompt.
- Each PromptSet must focus on one specific visual element of the class, do NOT mix in different elements of the class. That means only one specific elements can be used to to describe the effects in image_prompt and effect_prompt. This leads to only one element in effect_prompt but it can include more words to explain the single effect, but it should still be less than 2 sentences and cannot list multiple sub-effects within, you can only explain it using words. For example, if you are doing analog horror, the effects in both image_prompt and effect_prompt can only has one effect at a time, for example, "vhs decay" or "analog film grain", but not both or more in the same PromptSet. And you also cannot give a list like "perfect lens geometry, color-fringe-free edges, rectilinear perspective, professional optics, calibrated sensors" because that includes many elements. 
- The 10 sets should each highlight a different element or manifestation of the class."""

    user_msg = f"""Category: {top_key} > {cat_key}
Class: {class_name}
Description: {description}

Generate 10 PromptSet objects."""

    for attempt in range(3):
        try:
            response = client.beta.chat.completions.parse(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user_msg}
                ],
                response_format=ClassPrompts,
                extra_body={
                    "reasoning": {
                        "effort": "low"
                    }
                },
            )
            raw = response.choices[0].message.content
            return json.loads(raw)
        except Exception as e:
            print(f"  Attempt {attempt+1} failed for {class_name}: {e}")
            time.sleep(2)
    return None

# Filter out already-done classes
pending = [(t, c, n, d) for t, c, n, d in flat_classes
           if f"{t}/{c}/{n}" not in done]
print(f"Pending: {len(pending)}")

# Process with real-time JSONL writes — one line per prompt set (10 lines per class)
results_file = open(RESULTS_PATH, "a")
success_count = len(done)

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {}
    for top_key, cat_key, class_name, desc in pending:
        future = executor.submit(generate_prompts, top_key, cat_key, class_name, desc)
        futures[future] = (top_key, cat_key, class_name, desc)

    for future in tqdm.tqdm(as_completed(futures), total=len(futures), desc="Generating prompts"):
        top_key, cat_key, class_name, desc = futures[future]
        result = future.result()
        if result:
            base = {"top_category": top_key, "sub_category": cat_key, "class_name": class_name}
            for prompt_set in result["prompts"]:
                results_file.write(json.dumps({**base, **prompt_set}) + "\n")
            results_file.flush()
            success_count += 1
        else:
            print(f"FAILED: {top_key}/{cat_key}/{class_name}")

results_file.close()
print(f"\nDone. Total successful: {success_count}/{len(flat_classes)}")

Already done: 0/99
Pending: 99


Generating prompts:   0%|          | 0/99 [00:00<?, ?it/s]

Generating prompts: 100%|██████████| 99/99 [09:06<00:00,  5.52s/it]


Done. Total successful: 99/99


In [7]:
# Read back and preview
records = []
with open("generated_prompts.jsonl") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Total records: {len(records)}")
for val in records[:3]:
    print(f"\n{'='*60}")
    print(f"Class: {val['top_category']}/{val['sub_category']}/{val['class_name']}")
    print(f"  Image prompt:    {val['image_prompt'][:100]}...")
    print(f"  Negative prompt: {val['negative_prompt'][:100]}...")
    print(f"  Object prompt:   {val['object_prompt'][:100]}...")
    print(f"  Effects prompt:  {val['effects_prompt'][:100]}...")

Total records: 990

Class: anti_aesthetics/realism_and_style/plastic_toy_look
  Image prompt:    A small teddy bear sitting on a child's bedroom shelf, rendered with a uniform cheap plastic sheen o...
  Negative prompt: soft fabric texture, fuzzy fibers, matte finish, nuanced material variation, lifelike wear, tactile ...
  Object prompt:   A small teddy bear sitting on a child's bedroom shelf....
  Effects prompt:  Uniform cheap plastic sheen covering every surface, mass-produced synthetic finish with emotionally ...

Class: anti_aesthetics/realism_and_style/plastic_toy_look
  Image prompt:    A yellow duck floating in a bathtub, depicted with simplified molded details like a discount bath to...
  Negative prompt: fine feather detail, natural anatomy, intricate texture, realistic water interaction, organic surfac...
  Object prompt:   A yellow duck floating in a bathtub....
  Effects prompt:  Simplified molded detail with reduced surface information, blunt artificial shaping like a lo